# 00 — Monitor Oracle IoT

Visão geral e dashboard interativo das tabelas Oracle do projeto.

| Célula | Descrição |
|--------|-----------|
| **0** | Configuração e teste de conexão |
| **1** | Visão geral simultânea: Training + Monitoring |
| **2** | Dashboard interativo (browse detalhado por tabela) |
| **3** | Balanço de coleta por classe (guia de treinamento) |

> Execute **Run All** ou rode as células em ordem.

In [1]:
# =============================================================================
# Célula 0 — Configuração e teste de conexão
# =============================================================================
import os
import threading
from datetime import datetime

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sqlalchemy import create_engine, text

ORACLE_USER     = os.environ.get('ORACLE_USER',         'dersao')
ORACLE_PASSWORD = os.environ.get('ORACLE_PASSWORD',     '986960440')
ORACLE_HOST     = os.environ.get('ORACLE_HOST',         'localhost')
ORACLE_PORT     = os.environ.get('ORACLE_PORT',         '1521')
ORACLE_SERVICE  = os.environ.get('ORACLE_SERVICE_NAME', 'xepdb1')

ORACLE_CONN_STR = (
    f'oracle+oracledb://{ORACLE_USER}:{ORACLE_PASSWORD}'
    f'@{ORACLE_HOST}:{ORACLE_PORT}/?service_name={ORACLE_SERVICE}'
)

TABLE_TRAINING   = 'sensor_training_data'
TABLE_MONITORING = 'sensor_monitoring_data'
TABLE_LEGACY     = 'sensor_data'

_engine = create_engine(ORACLE_CONN_STR)

def _read_sql(query, conn):
    df = pd.read_sql(text(query), conn)
    df.columns = df.columns.str.lower()
    return df

print('=== Conexão Oracle ===')
print(f'Host: {ORACLE_HOST}:{ORACLE_PORT}  Service: {ORACLE_SERVICE}  Usuário: {ORACLE_USER}')
print()
for _t in (TABLE_TRAINING, TABLE_MONITORING, TABLE_LEGACY):
    try:
        with _engine.connect() as _c:
            _cnt = _c.execute(text(f'SELECT COUNT(*) FROM {_t}')).fetchone()[0]
        print(f'  {_t:35s} → {_cnt:>10,} registros')
    except Exception as _ex:
        print(f'  {_t:35s} → NÃO EXISTE')
print('\n✓ Pronto. Execute as próximas células.')


=== Conexão Oracle ===
Host: localhost:1521  Service: xepdb1  Usuário: dersao

  sensor_training_data                →          0 registros
  sensor_monitoring_data              →     10,000 registros
  sensor_data                         →          0 registros

✓ Pronto. Execute as próximas células.


In [2]:
# =============================================================================
# Célula 1 — Visão geral simultânea: Training + Monitoring
# Mostra resumo das DUAS tabelas ao mesmo tempo, com botão de refresh.
# =============================================================================

_SEP = '─' * 72

def _style_df(df):
    return (
        df.style
        .set_properties(**{'font-size': '12px', 'text-align': 'right'})
        .set_table_styles([{
            'selector': 'th',
            'props': [('font-size', '12px'), ('text-align', 'center'),
                      ('background-color', '#1a1a2e'), ('color', '#e2e8f0')]
        }])
        .hide(axis='index')
        .format(na_rep='—')
    )

def _now_epoch():
    return datetime.utcnow().timestamp()

def _overview_training(conn):
    """Resumo da sensor_training_data."""
    try:
        df = _read_sql(f"""
            SELECT
                COUNT(*)                                              AS registros,
                SUM(CASE WHEN transition_marker=0 THEN 1 ELSE 0 END) AS validos,
                COUNT(DISTINCT collection_id)                         AS colecoes,
                COUNT(DISTINCT NVL(cmd_speed_label, fan_state))       AS classes,
                ROUND(AVG(sample_rate), 1)                            AS hz_medio,
                ROUND((MAX(ts_epoch) - MIN(ts_epoch)) / 60, 1)        AS duracao_total_min,
                ROUND({_now_epoch()} - MAX(ts_epoch), 0)              AS ultimo_dado_ha_s
            FROM {TABLE_TRAINING}
        """, conn)
        return df
    except Exception as ex:
        return pd.DataFrame([{'erro': str(ex)}])

def _overview_monitoring(conn):
    """Resumo da sensor_monitoring_data."""
    try:
        df = _read_sql(f"""
            SELECT
                COUNT(*)                            AS registros,
                COUNT(DISTINCT collection_id)        AS sessoes,
                COUNT(DISTINCT predicted_class)      AS classes_preditas,
                ROUND(AVG(sample_rate), 1)           AS hz_medio,
                ROUND(AVG(confidence) * 100, 1)      AS confianca_media_pct,
                ROUND((MAX(ts_epoch) - MIN(ts_epoch)) / 60, 1) AS duracao_total_min,
                ROUND({_now_epoch()} - MAX(ts_epoch), 0)        AS ultimo_dado_ha_s
            FROM {TABLE_MONITORING}
        """, conn)
        return df
    except Exception as ex:
        return pd.DataFrame([{'erro': str(ex)}])

def _dist_training(conn):
    """Distribuição por classe na training."""
    try:
        return _read_sql(f"""
            SELECT
                NVL(cmd_speed_label, fan_state)                       AS velocidade,
                NVL(rot_state_label, '—')                             AS rotacao,
                SUM(CASE WHEN transition_marker=0 THEN 1 ELSE 0 END)  AS validos,
                COUNT(*)                                               AS total,
                ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)    AS pct
            FROM {TABLE_TRAINING}
            GROUP BY NVL(cmd_speed_label, fan_state), NVL(rot_state_label, '—')
            ORDER BY total DESC
        """, conn)
    except Exception as ex:
        return pd.DataFrame([{'erro': str(ex)}])

def _dist_monitoring(conn):
    """Distribuição de predições na monitoring."""
    try:
        return _read_sql(f"""
            SELECT
                NVL(predicted_class, '—')                          AS classe_predita,
                COUNT(*)                                           AS predicoes,
                ROUND(AVG(confidence) * 100, 1)                    AS conf_media_pct,
                ROUND(MIN(confidence) * 100, 1)                    AS conf_min_pct,
                ROUND(MAX(confidence) * 100, 1)                    AS conf_max_pct,
                ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
            FROM {TABLE_MONITORING}
            GROUP BY NVL(predicted_class, '—')
            ORDER BY predicoes DESC
        """, conn)
    except Exception as ex:
        return pd.DataFrame([{'erro': str(ex)}])

def _recent_monitoring(conn, n=10):
    """Últimos N registros da monitoring."""
    try:
        return _read_sql(f"""
            SELECT * FROM (
                SELECT
                    id,
                    TO_CHAR(
                        TIMESTAMP '1970-01-01 00:00:00 UTC'
                        + NUMTODSINTERVAL(ts_epoch - 10800, 'SECOND'),
                        'DD/MM HH24:MI:SS'
                    )                         AS data_hora,
                    ROUND(accel_x_g,  4)      AS ax_g,
                    ROUND(accel_y_g,  4)      AS ay_g,
                    ROUND(accel_z_g,  4)      AS az_g,
                    ROUND(gyro_x_dps, 2)      AS gx_dps,
                    ROUND(gyro_y_dps, 2)      AS gy_dps,
                    ROUND(gyro_z_dps, 2)      AS gz_dps,
                    NVL(predicted_class, '—') AS classe,
                    ROUND(confidence*100, 1)  AS conf_pct,
                    model_id,
                    sample_rate               AS hz
                FROM {TABLE_MONITORING}
                ORDER BY id DESC
            ) WHERE ROWNUM <= {n}
        """, conn)
    except Exception as ex:
        return pd.DataFrame([{'erro': str(ex)}])

def _recent_training(conn, n=10):
    """Últimos N registros da training."""
    try:
        return _read_sql(f"""
            SELECT * FROM (
                SELECT
                    id,
                    TO_CHAR(
                        TIMESTAMP '1970-01-01 00:00:00 UTC'
                        + NUMTODSINTERVAL(ts_epoch - 10800, 'SECOND'),
                        'DD/MM HH24:MI:SS'
                    )                                  AS data_hora,
                    ROUND(accel_x_g,  4)               AS ax_g,
                    ROUND(accel_y_g,  4)               AS ay_g,
                    ROUND(accel_z_g,  4)               AS az_g,
                    ROUND(gyro_x_dps, 2)               AS gx_dps,
                    ROUND(gyro_y_dps, 2)               AS gy_dps,
                    ROUND(gyro_z_dps, 2)               AS gz_dps,
                    NVL(cmd_speed_label, fan_state)    AS velocidade,
                    rot_state_label                    AS rotacao,
                    transition_marker                  AS trans,
                    sample_rate                        AS hz,
                    collection_id
                FROM {TABLE_TRAINING}
                ORDER BY id DESC
            ) WHERE ROWNUM <= {n}
        """, conn)
    except Exception as ex:
        return pd.DataFrame([{'erro': str(ex)}])

# ---------- Widget de refresh ----------
_btn_ov   = widgets.Button(description='🔄 Refresh', button_style='primary',
                           layout=widgets.Layout(width='120px'))
_lbl_ov   = widgets.HTML('<span style="color:#888">—</span>')
_out_ov   = widgets.Output()

def _run_overview(_=None):
    ts = datetime.now().strftime('%H:%M:%S')
    with _out_ov:
        clear_output(wait=True)
        try:
            with _engine.connect() as conn:
                df_tr_sum  = _overview_training(conn)
                df_mon_sum = _overview_monitoring(conn)
                df_tr_dist = _dist_training(conn)
                df_mon_dist= _dist_monitoring(conn)
                df_tr_rec  = _recent_training(conn)
                df_mon_rec = _recent_monitoring(conn)

            # ── TRAINING ──────────────────────────────────────────────────
            display(HTML(f'<h3 style="color:#60a5fa;margin:12px 0 4px">📦 TRAINING — sensor_training_data</h3>'))
            print(_SEP)
            print('  RESUMO')
            print(_SEP)
            display(_style_df(df_tr_sum))

            print()
            print(_SEP)
            print('  DISTRIBUIÇÃO POR CLASSE  (validos / total)')
            print(_SEP)
            display(_style_df(df_tr_dist))

            print()
            print(_SEP)
            print('  ÚLTIMOS 10 REGISTROS')
            print(_SEP)
            display(_style_df(df_tr_rec))

            # ── MONITORING ────────────────────────────────────────────────
            display(HTML(f'<h3 style="color:#34d399;margin:20px 0 4px">📡 MONITORING — sensor_monitoring_data</h3>'))
            print(_SEP)
            print('  RESUMO')
            print(_SEP)
            display(_style_df(df_mon_sum))

            print()
            print(_SEP)
            print('  DISTRIBUIÇÃO POR CLASSE PREDITA')
            print(_SEP)
            if df_mon_dist.empty or 'erro' in df_mon_dist.columns:
                print('  Nenhuma predição registrada ainda.')
            else:
                display(_style_df(df_mon_dist))

            print()
            print(_SEP)
            print('  ÚLTIMOS 10 REGISTROS')
            print(_SEP)
            if df_mon_rec.empty or 'erro' in df_mon_rec.columns:
                print('  Nenhum dado de monitoramento ainda.')
            else:
                display(_style_df(df_mon_rec))

            _lbl_ov.value = f'<span style="color:#00c896">✓ Atualizado: <b>{ts}</b></span>'
        except Exception as ex:
            _lbl_ov.value = f'<span style="color:#ff4444">Erro: {ex}</span>'
            print(f'Erro: {ex}')

_btn_ov.on_click(_run_overview)
display(widgets.HBox([_btn_ov, _lbl_ov]))
display(_out_ov)
_run_overview()


Output()

In [3]:
# =============================================================================
# Celula 1 — Dashboard interativo com suporte dual de tabelas
# =============================================================================
from IPython.display import HTML

display(HTML("""
<style>
.sticky-refresh {
    position: sticky !important;
    top: 0 !important;
    z-index: 1000 !important;
    background-color: #111827 !important;
    padding: 6px 4px !important;
    border-bottom: 1px solid #2d3748 !important;
    margin-bottom: 6px !important;
}
</style>
"""))

# Limpa instancia anterior
_prev = globals().get('_monitor_dashboard_state')
if _prev:
    try: _prev['stop_event'].set()
    except Exception: pass
    try:
        _t = _prev.get('thread')
        if _t and _t.is_alive(): _t.join(timeout=0.5)
    except Exception: pass
    for _w in _prev.get('widgets', []):
        try: _w.close()
        except Exception: pass

# ---------- Controles ----------
dropdown_table = widgets.Dropdown(
    options=[
        ('Treino  (sensor_training_data)',   TABLE_TRAINING),
        ('Monitor (sensor_monitoring_data)', TABLE_MONITORING),
        ('Legado  (sensor_data)',             TABLE_LEGACY),
    ],
    value=TABLE_TRAINING,
    description='Tabela:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='340px'),
)
btn_refresh = widgets.Button(
    description='Refresh', button_style='primary', icon='refresh',
    layout=widgets.Layout(width='110px', height='32px'),
)
toggle_auto = widgets.ToggleButton(
    value=False, description='Auto OFF', button_style='', icon='clock-o',
    layout=widgets.Layout(width='110px', height='32px'),
)
dropdown_interval = widgets.Dropdown(
    options=[('5s', 5), ('10s', 10), ('30s', 30), ('60s', 60)],
    value=5, description='Intervalo:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='150px'), disabled=True,
)
slider_rows = widgets.IntSlider(
    value=30, min=10, max=200, step=10, description='Linhas:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px'),
)
dropdown_collection = widgets.Dropdown(
    options=['(todas)'], value='(todas)', description='Collection:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='340px'),
)
lbl_status = widgets.HTML(value='<span style="color:#888">Aguardando primeiro refresh...</span>')
out_main   = widgets.Output()

_stop_event   = threading.Event()
_auto_thread  = None
_refresh_lock = threading.Lock()
_SEP          = '-' * 70

# ---------- Queries adaptativas ----------

def _fetch_collections(table):
    try:
        with _engine.connect() as conn:
            rows = conn.execute(text(
                f"SELECT collection_id FROM {table} "
                f"GROUP BY collection_id ORDER BY MAX(ts_epoch) DESC"
            )).fetchall()
        return ['(todas)'] + [r[0] for r in rows if r[0]]
    except Exception:
        return ['(todas)']


def _where_clause(collection):
    if collection == '(todas)':
        return ''
    safe = collection.replace("'", "''")
    return f"WHERE collection_id = \'{safe}\'"


def _fetch_training(conn, n_rows, where, table):
    """Queries para sensor_training_data / sensor_data (schema com labels)."""
    df_stats = _read_sql(f"""
        SELECT
            COUNT(*)                                        AS total_registros,
            MAX(id)                                         AS ultimo_id,
            ROUND(AVG(sample_rate), 1)                     AS hz_medio,
            ROUND((MAX(ts_epoch) - MIN(ts_epoch)) / 60, 1) AS duracao_total_min,
            ROUND(
                ((CAST(SYS_EXTRACT_UTC(SYSTIMESTAMP) AS DATE)
                  - DATE '1970-01-01') * 86400)
                - MAX(ts_epoch)
            , 1)                                            AS ultimo_envio_ha_s,
            SUM(CASE WHEN transition_marker = 0 THEN 1 ELSE 0 END) AS amostras_validas,
            COUNT(DISTINCT collection_id)                   AS num_colecoes,
            COUNT(DISTINCT device_id)                       AS num_dispositivos
        FROM {table} {where}
    """, conn)

    df_dist = _read_sql(f"""
        SELECT
            NVL(cmd_speed_label, fan_state)              AS velocidade,
            NVL(rot_state_label, '-')                    AS rotacao,
            SUM(CASE WHEN transition_marker=0 THEN 1 ELSE 0 END) AS validos,
            COUNT(*)                                     AS total,
            ROUND(COUNT(*) * 100.0
                  / SUM(COUNT(*)) OVER (), 1)            AS pct
        FROM {table} {where}
        GROUP BY NVL(cmd_speed_label, fan_state), NVL(rot_state_label, '-')
        ORDER BY total DESC
    """, conn)

    df_last = _read_sql(f"""
        SELECT * FROM (
            SELECT
                id,
                TO_CHAR(
                    TIMESTAMP '1970-01-01 00:00:00 UTC'
                    + NUMTODSINTERVAL(ts_epoch - 10800, 'SECOND'),
                    'DD/MM HH24:MI:SS'
                )                    AS data_hora,
                ROUND(accel_x_g,  4) AS ax_g,
                ROUND(accel_y_g,  4) AS ay_g,
                ROUND(accel_z_g,  4) AS az_g,
                ROUND(gyro_x_dps, 2) AS gx_dps,
                ROUND(gyro_y_dps, 2) AS gy_dps,
                ROUND(gyro_z_dps, 2) AS gz_dps,
                ROUND(temperature,2) AS temp_c,
                NVL(cmd_speed_label, fan_state) AS velocidade,
                rot_state_label      AS rotacao,
                transition_marker    AS trans,
                sample_rate          AS hz,
                rssi                 AS rssi_dbm,
                collection_id
            FROM {table} {where}
            ORDER BY id DESC
        ) WHERE ROWNUM <= {n_rows}
    """, conn)

    return df_stats, df_dist, df_last


def _fetch_monitoring(conn, n_rows, where):
    """Queries para sensor_monitoring_data (schema com predicao do modelo)."""
    df_stats = _read_sql(f"""
        SELECT
            COUNT(*)                                        AS total_registros,
            MAX(id)                                         AS ultimo_id,
            ROUND(AVG(sample_rate), 1)                     AS hz_medio,
            ROUND((MAX(ts_epoch) - MIN(ts_epoch)) / 60, 1) AS duracao_total_min,
            ROUND(
                ((CAST(SYS_EXTRACT_UTC(SYSTIMESTAMP) AS DATE)
                  - DATE '1970-01-01') * 86400)
                - MAX(ts_epoch)
            , 1)                                            AS ultimo_envio_ha_s,
            ROUND(AVG(confidence) * 100, 1)                AS confianca_media_pct,
            COUNT(DISTINCT collection_id)                   AS num_sessoes,
            COUNT(DISTINCT device_id)                       AS num_dispositivos
        FROM {TABLE_MONITORING} {where}
    """, conn)

    df_dist = _read_sql(f"""
        SELECT
            NVL(predicted_class, '-')            AS classe_predita,
            COUNT(*)                             AS predicoes,
            ROUND(AVG(confidence) * 100, 1)      AS confianca_media_pct,
            ROUND(MIN(confidence) * 100, 1)      AS confianca_min_pct,
            ROUND(MAX(confidence) * 100, 1)      AS confianca_max_pct,
            ROUND(COUNT(*) * 100.0
                  / SUM(COUNT(*)) OVER (), 1)    AS pct
        FROM {TABLE_MONITORING} {where}
        GROUP BY NVL(predicted_class, '-')
        ORDER BY predicoes DESC
    """, conn)

    df_last = _read_sql(f"""
        SELECT * FROM (
            SELECT
                id,
                TO_CHAR(
                    TIMESTAMP '1970-01-01 00:00:00 UTC'
                    + NUMTODSINTERVAL(ts_epoch - 10800, 'SECOND'),
                    'DD/MM HH24:MI:SS'
                )                        AS data_hora,
                ROUND(accel_x_g,  4)     AS ax_g,
                ROUND(accel_y_g,  4)     AS ay_g,
                ROUND(accel_z_g,  4)     AS az_g,
                ROUND(gyro_x_dps, 2)     AS gx_dps,
                ROUND(gyro_y_dps, 2)     AS gy_dps,
                ROUND(gyro_z_dps, 2)     AS gz_dps,
                NVL(predicted_class, '-') AS classe_predita,
                ROUND(confidence * 100, 1) AS confianca_pct,
                model_id,
                sample_rate              AS hz,
                collection_id
            FROM {TABLE_MONITORING} {where}
            ORDER BY id DESC
        ) WHERE ROWNUM <= {n_rows}
    """, conn)

    return df_stats, df_dist, df_last


def _fetch_all(n_rows, collection, table):
    where = _where_clause(collection)
    with _engine.connect() as conn:
        if table == TABLE_MONITORING:
            return _fetch_monitoring(conn, n_rows, where)
        else:
            return _fetch_training(conn, n_rows, where, table)


def _render_table(df, title):
    print('\n' + _SEP)
    print('  ' + title)
    print(_SEP)
    display(
        df.style
        .set_properties(**{'font-size': '12px', 'text-align': 'right'})
        .set_table_styles([{
            'selector': 'th',
            'props': [('font-size', '12px'), ('text-align', 'center'),
                      ('background-color', '#1a1a2e'), ('color', 'white')]
        }])
        .hide(axis='index')
        .format(na_rep='-')
    )

def _do_refresh():
    if not _refresh_lock.acquire(blocking=False):
        return
    try:
        n_rows     = slider_rows.value
        collection = dropdown_collection.value
        table      = dropdown_table.value
        ts         = datetime.now().strftime('%H:%M:%S')
        with out_main:
            clear_output(wait=True)
            try:
                df_stats, df_dist, df_last = _fetch_all(n_rows, collection, table)
                total    = int(df_stats['total_registros'].iloc[0])
                ultimo_s = df_stats['ultimo_envio_ha_s'].iloc[0]
                lbl_status.value = (
                    '<span style="color:#00c896">'
                    f'Atualizado: <b>{ts}</b> &nbsp;|&nbsp; '
                    f'Total: <b>{total:,}</b> &nbsp;|&nbsp; '
                    f'Ultimo envio ha <b>{ultimo_s}s</b> &nbsp;|&nbsp; '
                    f'<b>{table}</b>'
                    '</span>'
                )
                _render_table(df_stats, 'RESUMO GERAL')
                if table == TABLE_MONITORING:
                    _render_table(df_dist, 'DISTRIBUICAO POR CLASSE PREDITA')
                else:
                    _render_table(df_dist, 'DISTRIBUICAO POR CLASSE (validos / total)')
                _render_table(df_last, f'ULTIMOS {n_rows} REGISTROS (mais recente primeiro)')
            except Exception as e:
                lbl_status.value = f'<span style="color:#ff4444">Erro: {e}</span>'
                print(f'Erro ao carregar dados: {e}')
    finally:
        _refresh_lock.release()

def _auto_loop():
    while not _stop_event.is_set():
        _do_refresh()
        _stop_event.wait(timeout=dropdown_interval.value)

def on_refresh(_):
    _do_refresh()

def on_table_change(change):
    if change['name'] == 'value':
        new_table = change['new']
        cols = _fetch_collections(new_table)
        dropdown_collection.options = cols
        dropdown_collection.value   = cols[0]
        _do_refresh()

def on_toggle_auto(change):
    global _auto_thread
    if change['new']:
        toggle_auto.description    = 'Auto ON'
        toggle_auto.button_style   = 'success'
        dropdown_interval.disabled = False
        if _auto_thread and _auto_thread.is_alive():
            _stop_event.set()
            _auto_thread.join(timeout=0.5)
        _stop_event.clear()
        _auto_thread = threading.Thread(target=_auto_loop, daemon=True)
        _auto_thread.start()
        _monitor_dashboard_state['thread'] = _auto_thread
    else:
        toggle_auto.description    = 'Auto OFF'
        toggle_auto.button_style   = ''
        dropdown_interval.disabled = True
        _stop_event.set()
        if _auto_thread and _auto_thread.is_alive():
            _auto_thread.join(timeout=0.5)
        _monitor_dashboard_state['thread'] = _auto_thread

btn_refresh.on_click(on_refresh)
toggle_auto.observe(on_toggle_auto, names='value')
dropdown_table.observe(on_table_change, names='value')

# Popula colecoes para tabela inicial
dropdown_collection.options = _fetch_collections(TABLE_TRAINING)

toolbar = widgets.VBox([
    widgets.HBox(
        [btn_refresh, toggle_auto, dropdown_interval],
        layout=widgets.Layout(gap='10px', align_items='center'),
    ),
    widgets.HBox([dropdown_table, dropdown_collection],
                 layout=widgets.Layout(gap='10px', align_items='center')),
    widgets.HBox([slider_rows]),
    lbl_status,
])
toolbar.add_class('sticky-refresh')

_monitor_dashboard_state = {
    'stop_event': _stop_event,
    'thread': _auto_thread,
    'widgets': [toolbar, out_main, btn_refresh, toggle_auto,
                dropdown_interval, slider_rows, dropdown_collection,
                dropdown_table, lbl_status],
}

display(toolbar)
display(out_main)
_do_refresh()


Output()

In [ ]:
# =============================================================================
# Célula 3 — Balanço de coleta por classe (guia para treinamento)
# =============================================================================
TARGET_AMOSTRAS = 500

_S = '=' * 72

def _print_training_balance():
    with _engine.connect() as _conn:
        df_bal = _read_sql(f"""
            SELECT
                collection_id,
                NVL(cmd_speed_label, fan_state)              AS classe,
                COUNT(*)                                     AS amostras,
                SUM(CASE WHEN transition_marker=1 THEN 1 ELSE 0 END) AS transicao,
                SUM(CASE WHEN transition_marker=0 THEN 1 ELSE 0 END) AS validos,
                MIN(ts_epoch)                                AS inicio_epoch,
                ROUND((MAX(ts_epoch)-MIN(ts_epoch))/60, 1)   AS duracao_min
            FROM {TABLE_TRAINING}
            WHERE transition_marker IS NOT NULL
            GROUP BY collection_id, NVL(cmd_speed_label, fan_state)
            ORDER BY collection_id, classe
        """, _conn)

    if not df_bal.empty:
        _dt = pd.to_datetime(df_bal['inicio_epoch'], unit='s', utc=True, errors='coerce')
        df_bal['inicio'] = _dt.dt.tz_convert('America/Sao_Paulo').dt.strftime('%Y-%m-%d %H:%M')
    else:
        df_bal['inicio'] = []

    def _meta(v):
        v = int(v) if v else 0
        return f'OK ({v:,})' if v >= TARGET_AMOSTRAS else f'faltam {TARGET_AMOSTRAS - v}'

    df_bal['meta'] = df_bal['validos'].apply(_meta) if not df_bal.empty else []
    cols = ['collection_id', 'classe', 'amostras', 'transicao', 'validos', 'inicio', 'duracao_min', 'meta']

    print(_S)
    print(f'  BALANÇO DE TREINO [{TABLE_TRAINING}]  —  meta: {TARGET_AMOSTRAS} amostras válidas/classe')
    print(_S)
    if df_bal.empty:
        print('  Nenhum dado de treino encontrado.')
        return
    display(
        df_bal.reindex(columns=cols)
        .style
        .set_properties(**{'font-size': '12px', 'text-align': 'right'})
        .set_table_styles([{'selector': 'th', 'props': [
            ('font-size', '12px'), ('text-align', 'center'),
            ('background-color', '#1a1a2e'), ('color', 'white')]}])
        .map(lambda v: 'color:#00c896' if str(v).startswith('OK')
             else ('color:#ff4444' if str(v).startswith('faltam') else ''),
             subset=['meta'])
        .hide(axis='index')
        .format(na_rep='—')
    )


def _print_monitoring_summary():
    try:
        with _engine.connect() as _conn:
            df_mon = _read_sql(f"""
                SELECT
                    NVL(predicted_class, '—')       AS classe_predita,
                    COUNT(*)                        AS predicoes,
                    ROUND(AVG(confidence)*100, 1)   AS conf_media_pct,
                    ROUND(MIN(confidence)*100, 1)   AS conf_min_pct,
                    ROUND(MAX(confidence)*100, 1)   AS conf_max_pct,
                    COUNT(DISTINCT model_id)        AS modelos_usados,
                    COUNT(DISTINCT collection_id)   AS num_sessoes
                FROM {TABLE_MONITORING}
                GROUP BY NVL(predicted_class, '—')
                ORDER BY predicoes DESC
            """, _conn)
    except Exception as _ex:
        print(f'  Tabela {TABLE_MONITORING} não disponível: {_ex}')
        return

    print()
    print(_S)
    print(f'  RESUMO DE PREDIÇÕES [{TABLE_MONITORING}]')
    print(_S)
    if df_mon.empty:
        print('  Nenhuma predição registrada ainda.')
        return
    display(
        df_mon.style
        .set_properties(**{'font-size': '12px', 'text-align': 'right'})
        .set_table_styles([{'selector': 'th', 'props': [
            ('font-size', '12px'), ('text-align', 'center'),
            ('background-color', '#1a1a2e'), ('color', 'white')]}])
        .hide(axis='index')
        .format(na_rep='—')
    )


_print_training_balance()
_print_monitoring_summary()
